# 🌿 Eco-Vision: Improved Waste Classification Model
**Enhanced training pipeline with proper preprocessing, validation, and fine-tuning**

In [ ]:
# Install dependencies
!pip install -q kagglehub tensorflow scikit-learn matplotlib

import kagglehub
import tensorflow as tf
import numpy as np
import os, shutil, json
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ TensorFlow version:", tf.__version__)
print("✅ GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Download dataset from Kaggle
path = kagglehub.dataset_download("kaanerkez/waste-classfication-dataset")
base_path = os.path.join(path, "balanced_waste_images")

print("✅ Dataset location:", base_path)
print("✅ Classes:", os.listdir(base_path))

In [ ]:
# Copy to working directory
dst_path = "/content/waste_dataset"
shutil.copytree(base_path, dst_path, dirs_exist_ok=True)
base_path = dst_path

print("✅ Dataset ready at:", base_path)
print("✅ Total classes:", len(os.listdir(base_path)))

In [ ]:
# === IMPROVED: Proper train/val/test split (no data leakage) ===
train_dir = "/content/dataset/train"
val_dir = "/content/dataset/val"
test_dir = "/content/dataset/test"

for dir_path in [train_dir, val_dir, test_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Split: 70% train, 15% val, 15% test
for cls in os.listdir(base_path):
    cls_path = os.path.join(base_path, cls)
    if not os.path.isdir(cls_path):
        continue
    
    images = os.listdir(cls_path)
    
    # First split: 70/30
    train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=42)
    # Second split: 50/50 of remaining (15% val, 15% test)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)
    
    # Create class folders
    for dir_path in [train_dir, val_dir, test_dir]:
        os.makedirs(os.path.join(dir_path, cls), exist_ok=True)
    
    # Copy images
    for img in train_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(train_dir, cls, img))
    for img in val_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(val_dir, cls, img))
    for img in test_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(test_dir, cls, img))

print("✅ Train/val/test split complete")
print(f"   Train: {len(os.listdir(train_dir))} classes")
print(f"   Val: {len(os.listdir(val_dir))} classes")
print(f"   Test: {len(os.listdir(test_dir))} classes")

In [ ]:
# === IMPROVED: Better data loading with augmentation ===
img_size = (224, 224)
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE

# Training data: aggressive augmentation
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int',
    shuffle=True,
    seed=42
)

# Validation data: no augmentation, shuffle=False
val_data = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int',
    shuffle=False
)

# Test data: no augmentation, shuffle=False
test_data = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int',
    shuffle=False
)

class_names = train_data.class_names
print(f"✅ Classes ({len(class_names)}):")
print(class_names)

# Cache & prefetch
train_data = train_data.cache().shuffle(1000).prefetch(AUTOTUNE)
val_data = val_data.cache().prefetch(AUTOTUNE)
test_data = test_data.cache().prefetch(AUTOTUNE)

In [ ]:
# === IMPROVED: Proper MobileNetV2 preprocessing + better augmentation ===
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = tf.keras.Sequential([
    # === IMPROVED: Use proper preprocess_input instead of Rescaling ===
    tf.keras.layers.Lambda(lambda x: preprocess_input(x)),
    
    # Enhanced data augmentation
    tf.keras.layers.RandomFlip("horizontal", seed=42),
    tf.keras.layers.RandomRotation(0.25, seed=42),
    tf.keras.layers.RandomZoom(0.2, seed=42),
    tf.keras.layers.RandomContrast(0.2, seed=42),
    tf.keras.layers.RandomBrightness(0.2, seed=42),
    
    # Base model
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    
    # Improved dense layers
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    
    # Output layer
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

print("✅ Model architecture:")
model.summary()

In [ ]:
# === IMPROVED: Label smoothing + better loss ===
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

print("✅ Model compiled with label smoothing")

In [ ]:
# === IMPROVED: Better callbacks ===
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.h5",
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    )
]

print("✅ Callbacks configured")

In [ ]:
# === Phase 1: Train frozen base model ===
print("🔄 Phase 1: Training frozen base model (10 epochs)...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

print("✅ Phase 1 complete")

In [ ]:
# === IMPROVED: Fine-tune deeper (more layers) ===
print("🔄 Phase 2: Fine-tuning (unfreezing more layers)...")

base_model.trainable = True

# Unfreeze last 80 layers (was 20 in original)
for layer in base_model.layers[:-80]:
    layer.trainable = False

print(f"✅ Unfrozen {sum(1 for l in base_model.layers[-80:] if l.trainable)} layers")

# Lower learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

print("✅ Model recompiled with lower learning rate")

In [ ]:
# === Phase 2: Fine-tune with unfrozen layers ===
print("🔄 Phase 2: Fine-tuning (8 epochs)...")
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=8,
    callbacks=callbacks,
    verbose=1
)

print("✅ Phase 2 complete")

In [ ]:
# === Evaluate on TEST set (not validation!) ===
print("📊 Evaluating on TEST set...")
test_loss, test_acc = model.evaluate(test_data, verbose=1)
print(f"\n✅ Test Accuracy: {test_acc*100:.2f}%")
print(f"✅ Test Loss: {test_loss:.4f}")

In [ ]:
# === Detailed evaluation metrics ===
print("🔍 Computing detailed metrics...")

# Get predictions
y_true = []
y_pred = []

for images, labels in test_data:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Classification report
print("\n📈 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# F1 score
f1 = f1_score(y_true, y_pred, average='weighted')
print(f"\n✅ Weighted F1 Score: {f1:.4f}")

In [ ]:
# === Confusion Matrix ===
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("✅ Confusion matrix plotted")

In [ ]:
# === IMPORTANT: Export model + class_names ===
# Save the model
model.save('waste_classifier_model.h5')
print("✅ Model saved: waste_classifier_model.h5")

# Save class names in exact order
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print("✅ Class names saved: class_names.json")
print(f"\n📋 Class order (CRITICAL for backend):")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

In [ ]:
# === Download files for backend ===
from google.colab import files

print("📥 Downloading model files...")
files.download('waste_classifier_model.h5')
files.download('class_names.json')
print("✅ Download started (check your Downloads folder)")

In [ ]:
# === Test predictions ===
def predict_and_show(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    # Note: preprocess_input is applied in the model
    
    pred = model.predict(img_array, verbose=0)
    
    top_3_idx = np.argsort(pred[0])[::-1][:3]
    
    print(f"\n📸 Predictions for: {img_path}")
    for idx in top_3_idx:
        print(f"  {class_names[idx]}: {pred[0][idx]*100:.1f}%")
    
    plt.imshow(img)
    plt.title(f"Top Prediction: {class_names[top_3_idx[0]]}")
    plt.axis('off')
    plt.show()

print("✅ Prediction function ready")

In [ ]:
# === Upload & test image ===
from google.colab import files

print("📤 Upload image to test:")
uploaded = files.upload()

for img_name in uploaded.keys():
    predict_and_show(img_name)